In [0]:
%python
dbutils.widgets.removeAll()

In [0]:
create widget text storageName default "saejercdatabricks";

In [0]:
%python
storageName = dbutils.widgets.get("storageName")

In [0]:
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-metastore`
URL 'abfss://metastore@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `ejerc_databrics_credential`)
COMMENT 'Ubicación externa para las tablas raw del Data Lake';

In [0]:
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-ejerc_raw`
URL 'abfss://raw@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `ejerc_databrics_credential`)
COMMENT 'Ubicación externa de las tablas raw del Data Lake';

In [0]:
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-ejerc_bronze`
URL 'abfss://bronze@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `ejerc_databrics_credential`)
COMMENT 'Ubicación externa para las tablas bronze del Data Lake';

In [0]:
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-ejerc_silver`
URL 'abfss://silver@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `ejerc_databrics_credential`)
COMMENT 'Ubicación externa para las tablas silver del Data Lake';

In [0]:
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-ejerc_golden`
URL 'abfss://golden@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `ejerc_databrics_credential`)
COMMENT 'Ubicación externa para las tablas golden del Data Lake';

In [0]:
-- DESCRIBE EXTERNAL LOCATION `exlt-ejerc_bronze`;
SHOW EXTERNAL LOCATIONS;

In [0]:
DROP CATALOG IF EXISTS cat_ejec_dbricks CASCADE;

In [0]:
CREATE CATALOG IF NOT EXISTS cat_ejec_dbricks
MANAGED LOCATION 'abfss://metastore@${storageName}.dfs.core.windows.net/'
COMMENT 'Catalogo para la arquitectura medallion del ambiente Dev';

In [0]:
DROP SCHEMA IF EXISTS cat_ejec_dbricks.raw;
DROP SCHEMA IF EXISTS cat_ejec_dbricks.bronze;
DROP SCHEMA IF EXISTS cat_ejec_dbricks.silver;
DROP SCHEMA IF EXISTS cat_ejec_dbricks.golden;

In [0]:
-- CREATE SCHEMA IF NOT EXISTS cat_ejec_dbricks.raw;
CREATE SCHEMA IF NOT EXISTS cat_ejec_dbricks.bronze;
CREATE SCHEMA IF NOT EXISTS cat_ejec_dbricks.silver;
CREATE SCHEMA IF NOT EXISTS cat_ejec_dbricks.golden;

###Tablas Bronze

In [0]:
CREATE TABLE cat_ejec_dbricks.bronze.suspensiones (
codigo Integer,
descripcion string,
observacion string,
fecha_grabada timestamp,
fecha_realizada timestamp,
fecrea_fin timestamp,
grupo string,
suscriptor integer,
ciclo string,
estrato integer,
finalizada_por string,
deuda integer,
ingestion_date timestamp
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/suspensiones"

In [0]:
CREATE TABLE cat_ejec_dbricks.bronze.reinstalaciones (
codigo integer,
descripcion string,
observacion string,
fecha_grabada TIMESTAMP,
fecha_planificacion TIMESTAMP,
fecha_realizada TIMESTAMP,
suscriptor integer,
zona string,
ciclo string,
estrato integer,
cobro string,
resultado string,
usu_realiza string,
ult_fecpago string,
ingestion_date timestamp
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/reinstalaciones"

###Tablas Silver

In [0]:
CREATE TABLE IF NOT EXISTS cat_ejec_dbricks.silver.suspen_reinsta (
SUSCRIPTOR STRING,
CICLO STRING,
ESTRATO STRING,
PERIODO STRING,
DEUDA INTEGER,
DIAS_EJEC_SUSP INTEGER,
DIAS_EJEC_REINS INTEGER,
DIAS_REINSTALACION INTEGER,
DIAS_MORA INTEGER,
VECES INTEGER,
INGESTION_DATE timestamp
)
USING DELTA
LOCATION "abfss://silver@${storageName}.dfs.core.windows.net/suspen_reinsta"

###Tablas Golden

In [0]:
CREATE TABLE IF NOT EXISTS cat_ejec_dbricks.golden.suspen_reinsta_resultados (
ESTRATO STRING,
CICLO STRING,
PERIODO STRING,
MAX_DEUDA INTEGER,
AVG_DEUDA INTEGER,
AVG_DIAS_EJEC_SUSP INTEGER,
AVG_DIAS_EJEC_REINS INTEGER,
MIN_TIEMPO_REINST INTEGER,
MAX_TIEMPO_REINST INTEGER,
MORA_MAXIMA INTEGER,
TOTAL_SUSPENSIONES INTEGER
)
USING DELTA
LOCATION "abfss://golden@${storageName}.dfs.core.windows.net/suspen_reinsta_resultados"